# 03 Modeling

This notebook trains and compares machine learning regression models for predicting airport security checkpoint processing time.

In [114]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [115]:
file_path = "../data/processed/security_processed.csv"
df = pd.read_csv(file_path)

print(df.shape)
df.head()

(2396, 52)


,Comments,Lane number,Date,business,senior,family,young,PRM,regular,Experience,...,collect_time,patdown_time,etd_time,baggage_check_time,has_second_wtmd,has_third_wtmd,has_patdown,has_etd,has_baggage_check,passenger_type
0,"2x wtmd, shoes",3,2/23/2018 0:00,0,0,0,0,0,1,0,...,100.0,NaN,NaN,NaN,1,0,0,0,0,regular
1,NaN,3,2/23/2018 0:00,0,0,0,0,0,1,0,...,103.0,NaN,7.0,NaN,0,0,0,1,0,regular
2,NaN,3,2/23/2018 0:00,0,0,0,0,0,1,0,...,39.0,NaN,NaN,NaN,0,0,0,0,0,regular
3,2x wtmd,3,2/23/2018 0:00,0,0,0,0,0,1,0,...,61.0,NaN,7.0,NaN,1,0,0,1,0,regular
4,NaN,3,2/23/2018 0:00,1,0,0,0,0,1,1,...,58.0,NaN,NaN,NaN,0,0,0,0,0,business


In [116]:
df.columns.tolist()

['Comments',
 'Lane number',
 'Date',
 'business',
 'senior',
 'family',
 'young',
 'PRM',
 'regular',
 'Experience',
 'Time Start Baggage drop off',
 'Number of boxes',
 'Time End Baggage drop off',
 'Time Start WTMD',
 'Time Start WTMD 2',
 'Time Start WTMD 3',
 'Time Start WTMD check',
 'Time End WTMD check',
 'Time Start ETD check',
 'Time End ETD check',
 'Time Start Baggage reclaim',
 'Time End Baggage reclaim',
 'Group size',
 'Time Start Baggage Check',
 'Time End Baggage Check',
 'source_sheet',
 'Time Start Baggage drop off_sec',
 'Time End Baggage drop off_sec',
 'Time Start WTMD_sec',
 'Time Start WTMD 2_sec',
 'Time Start WTMD 3_sec',
 'Time Start WTMD check_sec',
 'Time End WTMD check_sec',
 'Time Start ETD check_sec',
 'Time End ETD check_sec',
 'Time Start Baggage reclaim_sec',
 'Time End Baggage reclaim_sec',
 'Time Start Baggage Check_sec',
 'Time End Baggage Check_sec',
 'checkpoint_time',
 'drop_time',
 'wait_I',
 'collect_time',
 'patdown_time',
 'etd_time',
 'bagg

In [117]:
feature_cols = [
    "Lane number",
    "Experience",
    "Number of boxes",
    "Group size",
    "business",
    "senior",
    "family",
    "young",
    "PRM",
    "regular"
]

target_col = "checkpoint_time"

In [118]:
model_df = df[feature_cols + [target_col]].dropna().copy()

print(model_df.shape)
model_df.head()

(2396, 11)


,Lane number,Experience,Number of boxes,Group size,business,senior,family,young,PRM,regular,checkpoint_time
0,3,0,1,2,0,0,0,0,0,1,204
1,3,0,2,2,0,0,0,0,0,1,223
2,3,0,1,2,0,0,0,0,0,1,80
3,3,0,2,2,0,0,0,0,0,1,125
4,3,1,1,1,1,0,0,0,0,1,125


In [119]:
X = model_df[feature_cols]
y = model_df[target_col]

print(X.shape)
print(y.shape)

(2396, 10)
(2396,)


In [120]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (1916, 10)
X_test: (480, 10)
y_train: (1916,)
y_test: (480,)


In [121]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [122]:
y_pred_lr = lr_model.predict(X_test)

In [123]:
mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

print("Linear Regression Results")
print(f"MAE:  {mae_lr:.2f}")
print(f"RMSE: {rmse_lr:.2f}")
print(f"R²:   {r2_lr:.4f}")

Linear Regression Results
MAE:  50.06
RMSE: 65.64
R²:   0.1099


# Random Forest

In [124]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [125]:
y_pred_rf = rf_model.predict(X_test)

In [126]:
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print("Random Forest Results")
print(f"MAE:  {mae_rf:.2f}")
print(f"RMSE: {rmse_rf:.2f}")
print(f"R²:   {r2_rf:.4f}")

Random Forest Results
MAE:  49.02
RMSE: 65.01
R²:   0.1270


In [127]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "MAE": [mae_lr, mae_rf],
    "RMSE": [rmse_lr, rmse_rf],
    "R2": [r2_lr, r2_rf]
})

results

,Model,MAE,RMSE,R2
0,Linear Regression,50.055434,65.640836,0.109864
1,Random Forest,49.020569,65.006753,0.126978


In [128]:
feature_importance = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": rf_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

feature_importance

,Feature,Importance
1,Experience,0.227892
3,Group size,0.204857
2,Number of boxes,0.147733
0,Lane number,0.087939
6,family,0.070972
9,regular,0.070941
7,young,0.058124
5,senior,0.055219
4,business,0.051564
8,PRM,0.024757


# Enrich data for linear


In [129]:
feature_cols_enriched = [
    "Lane number",
    "Experience",
    "Number of boxes",
    "Group size",
    "business",
    "senior",
    "family",
    "young",
    "PRM",
    "regular",
    "drop_time",
    "wait_I",
    "collect_time",
    "has_second_wtmd",
    "has_third_wtmd",
    "has_patdown",
    "has_etd",
    "has_baggage_check"
]

In [130]:
model_df_enriched = df[feature_cols_enriched + [target_col]].dropna().copy()

print(model_df_enriched.shape)
model_df_enriched.head()

(2390, 19)


,Lane number,Experience,Number of boxes,Group size,business,senior,family,young,PRM,regular,drop_time,wait_I,collect_time,has_second_wtmd,has_third_wtmd,has_patdown,has_etd,has_baggage_check,checkpoint_time
0,3,0,1,2,0,0,0,0,0,1,78.0,9.0,100.0,1,0,0,0,0,204
1,3,0,2,2,0,0,0,0,0,1,88.0,17.0,103.0,0,0,0,1,0,223
2,3,0,1,2,0,0,0,0,0,1,25.0,2.0,39.0,0,0,0,0,0,80
3,3,0,2,2,0,0,0,0,0,1,29.0,4.0,61.0,1,0,0,1,0,125
4,3,1,1,1,1,0,0,0,0,1,46.0,3.0,58.0,0,0,0,0,0,125


In [131]:
X_enriched = model_df_enriched[feature_cols_enriched]
y_enriched = model_df_enriched[target_col]

In [132]:
X_train_e, X_test_e, y_train_e, y_test_e = train_test_split(
    X_enriched, y_enriched, test_size=0.2, random_state=42
)

print("X_train_e:", X_train_e.shape)
print("X_test_e:", X_test_e.shape)
print("y_train_e:", y_train_e.shape)
print("y_test_e:", y_test_e.shape)

X_train_e: (1912, 18)
X_test_e: (478, 18)
y_train_e: (1912,)
y_test_e: (478,)


In [133]:
lr_model_e = LinearRegression()
lr_model_e.fit(X_train_e, y_train_e)

y_pred_lr_e = lr_model_e.predict(X_test_e)

mae_lr_e = mean_absolute_error(y_test_e, y_pred_lr_e)
rmse_lr_e = np.sqrt(mean_squared_error(y_test_e, y_pred_lr_e))
r2_lr_e = r2_score(y_test_e, y_pred_lr_e)

print("Enriched Linear Regression Results")
print(f"MAE:  {mae_lr_e:.2f}")
print(f"RMSE: {rmse_lr_e:.2f}")
print(f"R²:   {r2_lr_e:.4f}")

Enriched Linear Regression Results
MAE:  14.32
RMSE: 20.23
R²:   0.9229


# Random Forest

In [134]:
rf_model_e = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf_model_e.fit(X_train_e, y_train_e)

y_pred_rf_e = rf_model_e.predict(X_test_e)

mae_rf_e = mean_absolute_error(y_test_e, y_pred_rf_e)
rmse_rf_e = np.sqrt(mean_squared_error(y_test_e, y_pred_rf_e))
r2_rf_e = r2_score(y_test_e, y_pred_rf_e)

print("Enriched Random Forest Results")
print(f"MAE:  {mae_rf_e:.2f}")
print(f"RMSE: {rmse_rf_e:.2f}")
print(f"R²:   {r2_rf_e:.4f}")

Enriched Random Forest Results
MAE:  17.15
RMSE: 24.58
R²:   0.8862


In [137]:
comparison_results = pd.DataFrame({
    "Model": [
        "Baseline Linear Regression",
        "Baseline Random Forest",
        "Enriched Linear Regression",
        "Enriched Random Forest"
    ],
    "MAE": [
        mae_lr,
        mae_rf,
        mae_lr_e,
        mae_rf_e
    ],
    "RMSE": [
        rmse_lr,
        rmse_rf,
        rmse_lr_e,
        rmse_rf_e
    ],
    "R2": [
        r2_lr,
        r2_rf,
        r2_lr_e,
        r2_rf_e
    ]
})

comparison_results

,Model,MAE,RMSE,R2
0,Baseline Linear Regression,50.055434,65.640836,0.109864
1,Baseline Random Forest,49.020569,65.006753,0.126978
2,Enriched Linear Regression,14.316777,20.227120,0.922939
3,Enriched Random Forest,17.153138,24.577229,0.886228


In [139]:
feature_importance_enriched = pd.DataFrame({
    "Feature": feature_cols_enriched,
    "Importance": rf_model_e.feature_importances_
}).sort_values(by="Importance", ascending=False)

feature_importance_enriched

,Feature,Importance
12,collect_time,0.548139
10,drop_time,0.252256
11,wait_I,0.145517
3,Group size,0.007700
15,has_patdown,0.007397
17,has_baggage_check,0.006452
13,has_second_wtmd,0.005041
2,Number of boxes,0.004822
16,has_etd,0.003974
0,Lane number,0.003626
